In [2]:
import pandas as pd
import numpy as np

In [13]:
from sqlalchemy import create_engine

engine = create_engine('mssql+pyodbc://@LAPTOP/AD-Campaigns?driver=SQL+Server&Trusted_Connection=yes', fast_executemany=True, connect_args={"TrustServerCertificate": "yes"})

query = """
select 
    user_id,
    a.ad_id,
    timestamp,
    event_type,
    ad_platform
from ad_events a
join ads on a.ad_id = ads.ad_id
"""

df = pd.read_sql_query(query, engine)


In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df

,user_id,ad_id,timestamp,event_type,ad_platform
0,2359b,197,2025-07-26 00:19:56,Like,Facebook
1,f9c67,51,2025-06-15 08:28:07,Share,Instagram
2,5b868,46,2025-06-27 00:40:02,Impression,Instagram
3,3d440,166,2025-06-05 19:20:45,Impression,Instagram
4,68f1a,52,2025-07-22 08:30:29,Impression,Instagram
...,...,...,...,...,...
399995,3cb8c,132,2025-08-01 22:36:54,Impression,Facebook
399996,fe0e3,200,2025-05-31 14:53:18,Impression,Instagram
399997,a08c1,2,2025-07-27 13:39:51,Click,Facebook
399998,4f0cf,109,2025-05-16 02:38:23,Impression,Facebook


In [4]:
purchases = (
    df[df['event_type'] == 'Purchase']
    .sort_values(['user_id', 'timestamp'])
)

purchases['purchase_id'] = purchases.groupby('user_id').cumcount() + 1
# cumcount() = row_number()

purchases['prev_purchase_time'] = purchases.groupby('user_id')['timestamp'].shift(1)
# shift() = lag()

In [5]:
journey = df.merge(purchases[['user_id', 'purchase_id', 'timestamp', 'prev_purchase_time']], on='user_id', suffixes=('', '_purchase'))

journey = journey[
    (journey['timestamp'] < journey['timestamp_purchase']) & 
    (
        journey['timestamp'] > journey['prev_purchase_time'].fillna(pd.Timestamp('1900-01-01'))
    )
].copy()

journey = journey.rename(columns={'timestamp': 'event_time',
                                  'timestamp_purchase': 'purchase_date'})

In [7]:
# last-touch attribution

journey['rn'] = journey.groupby(['user_id', 'purchase_id'])['event_time'].rank(method='first', ascending=False)

last_touch = journey[journey['rn'] == 1]
last_touch_agg = (last_touch[last_touch['event_type'].isin(
        ['Click', 'Like', 'Comment', 'Share']
    )]
    .groupby(['ad_platform', 'event_type'])
    .size() # with gaps unlike count()
    .reset_index(name='last_touch_conversions')
)

In [8]:
# time-decay attribution

journey['time_diff_hours'] = (
    (journey['purchase_date'] - journey['event_time'])
    .dt.total_seconds() / 3600
)

In [9]:
half_life = 24

journey['td_raw_weight'] = 0.5 ** (
    journey['time_diff_hours'] / half_life
)
journey['td_weight'] = (
    journey['td_raw_weight'] /
    journey.groupby(['user_id', 'purchase_id'])['td_raw_weight'].transform('sum')
)

In [15]:
# linear attribution

path_length = (
    journey.groupby(['user_id', 'purchase_id'])
    .size()
    .reset_index(name='n')
)

journey['path_len'] = journey.groupby(['user_id', 'purchase_id'])['event_time'].transform('count')
journey['linear_weight'] = 1 / journey['path_len']

In [16]:
# position-based attribution

journey['pos'] = journey.groupby(['user_id', 'purchase_id']).cumcount() + 1

n = journey['path_len']
pos = journey['pos']

conditions = [
    (n == 1),
    (n == 2),
    (pos == 1),
    (pos == n)
]

choices = [
    1.0,
    0.5,
    0.4,
    0.4
]

journey['pos_weight'] = np.select(conditions, choices, default=np.where(n > 2, 0.2 / (n - 2), 0))

In [19]:
final = (
    journey
    .groupby(['ad_platform', 'event_type'])
    .agg(
        event_cnt=('event_type', 'count'),
        linear_conversions=('linear_weight', 'sum'),
        time_decay_conversions=('td_weight', 'sum'),
        position_conversions=('pos_weight', 'sum')
    )
    .reset_index()
)

final = final.merge(
    last_touch_agg,
    on=['ad_platform', 'event_type'],
    how='left'
)

final['last_touch_conversions'] = final['last_touch_conversions'].fillna(0)

In [20]:
final['td_vs_linear'] = final['time_decay_conversions'] / final['linear_conversions']
final['pos_vs_linear'] = final['position_conversions'] / final['linear_conversions']

In [21]:
final = final.sort_values('linear_conversions', ascending=False)

In [22]:
final

,ad_platform,event_type,event_cnt,linear_conversions,time_decay_conversions,position_conversions,last_touch_conversions,td_vs_linear,pos_vs_linear
2,Facebook,Impression,20582,1052.015232,1068.977911,1055.076811,0.0,1.016124,1.002910
7,Instagram,Impression,12003,621.978545,619.899557,608.697375,0.0,0.996657,0.978647
0,Facebook,Click,2446,127.136060,107.556729,127.714097,104.0,0.845997,1.004547
5,Instagram,Click,1486,77.095320,78.790993,80.427814,80.0,1.021994,1.043226
3,Facebook,Like,724,37.902625,35.038103,40.467942,29.0,0.924424,1.067682
8,Instagram,Like,382,18.560055,23.626490,20.886738,23.0,1.272975,1.125360
1,Facebook,Comment,255,12.542144,13.106347,13.549786,12.0,1.044985,1.080340
6,Instagram,Comment,153,7.497851,4.589608,7.175076,4.0,0.612123,0.956951
4,Facebook,Share,130,5.944675,8.686745,6.301479,10.0,1.461265,1.060021
9,Instagram,Share,62,3.327492,3.727519,3.702882,4.0,1.120219,1.112815


Markov Chain (data-driven)

In [23]:
import random
random.seed(42)
from collections import defaultdict, Counter

In [ ]:
# users successful paths

paths = (
    journey.sort_values(['user_id', 'purchase_id', 'event_time'])
    .groupby(['user_id', 'purchase_id'])['ad_platform']
    .apply(list)
    .reset_index(name='path')
)

paths['path'] = paths['path'].apply(lambda x: ['Start'] + x + ['Conversion'])

In [ ]:
# users unsuccessful paths

non_converted = df[~df['user_id'].isin(purchases['user_id'])]

paths_null = (
    non_converted
    .sort_values(['user_id', 'timestamp'])
    .groupby('user_id')['ad_platform']
    .apply(list)
    .reset_index(name='path')
)

paths_null['path'] = paths_null['path'].apply(lambda x: ['Start'] + x + ['Null'])

In [ ]:
paths_all = pd.concat([paths[['path']], paths_null[['path']]], ignore_index=True)

In [ ]:
# count transitions 

transitions = Counter()

for path in paths_all['path']:
    for i in range(len(path) - 1):
        transitions[(path[i], path[i+1])] += 1

In [ ]:
# transition matrix

trans_df = pd.DataFrame(
    [(k[0], k[1], v) for k, v in transitions.items()],
    columns=['from', 'to', 'count']
)

trans_df['prob'] = trans_df.groupby('from')['count'].transform(lambda x: x / x.sum())

In [ ]:
trans_dict = defaultdict(dict)

for _, row in trans_df.iterrows():
    trans_dict[row['from']][row['to']] = row['prob']

In [ ]:
# Monte Carlo model

def simulate(transitions, start='Start', n_sim=100000):

    success = 0
    absorbing = {'Conversion', 'Null'}

    for _ in range(n_sim):
        state = start

        while state not in absorbing:

            if state not in transitions or len(transitions[state]) == 0:
                state = 'Null'
                break

            next_states = transitions[state]
            states = list(next_states.keys())
            probs = list(next_states.values())

            state = random.choices(states, probs)[0]

        if state == 'Conversion':
            success += 1

    return success / n_sim

In [ ]:
def remove_channel(transitions, channel):

    new_trans = defaultdict(dict)

    for frm, to_dict in transitions.items():

        if frm == channel:
            continue

        adjusted = {}

        for to, p in to_dict.items():

            if to == channel:
                if channel in transitions:
                    for ch_next, ch_prob in transitions[channel].items():
                        adjusted[ch_next] = adjusted.get(ch_next, 0) + p * ch_prob
            else:
                adjusted[to] = adjusted.get(to, 0) + p

        total = sum(adjusted.values())

        if total > 0:
            adjusted = {k: v / total for k, v in adjusted.items()}

        new_trans[frm] = adjusted

    return new_trans

In [ ]:
base_prob = simulate(trans_dict)

In [ ]:
# Removal effect

channels = set(trans_df['from'].unique()) | set(trans_df['to'].unique())

removal_effects = {}

for ch in channels:

    if ch in ['Start', 'Conversion', 'Null']:
        continue

    modified = remove_channel(trans_dict, ch)

    prob = simulate(modified)

    removal_effects[ch] = base_prob - prob

In [ ]:
markov_df = pd.DataFrame(
    removal_effects.items(),
    columns=['ad_platform', 'removal_effect']
)

markov_df['removal_effect'] = markov_df['removal_effect'].clip(lower=0)

if markov_df['removal_effect'].sum() > 0:
    markov_df['attribution'] = (
        markov_df['removal_effect'] /
        markov_df['removal_effect'].sum()
    )
else:
    markov_df['attribution'] = 0

markov_df = markov_df.sort_values('attribution', ascending=False).reset_index(drop=True)

In [ ]:
markov_df

,ad_platform,removal_effect,attribution
0,Facebook,0.17784,0.535663
1,Instagram,0.15416,0.464337


In [15]:
user_conv = (
    df.groupby('user_id')['event_type']
    .apply(lambda x: (x == 'Purchase').sum() / len(x))
    .reset_index(name='user_conv_rate')
)

ad_conv = (
    df.groupby('ad_id')['event_type']
    .apply(lambda x: (x == 'Purchase').sum() / len(x))
    .reset_index(name='ad_conv_rate')
)

user_conv.to_csv('user_conv.csv', index=False)
ad_conv.to_csv('ad_conv.csv', index=False)